# 11.6 對齊和 SFT 能合併嗎？

# 第 11 章：對齊、問答與語言能力保留

前置：第10章介面與第7章SFT。能力實驗需先有「文字屬性→正確答案」基模，以及可分類的視覺encoder；用scripts/prepare_data.py和pretrain_encoders.py準備。下面隨機模型只驗證資料／梯度，所有品質留給讀者訓練後量測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：轉接頭尺寸合了，還要學資料的意思**

先確認文字基模會用文字屬性回答，再學圖片接頭。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/alignment.svg")))

**先想一想：**兩階段跑兩倍tokens，結果較好就證明順序更好嗎？

兩階段先caption對齊，再問答；直接SFT一次學接口與任務。比較時總預算要包含第一階段，而不是免費送給兩階段。

**把直覺寫成數學：**budget_two_stage=tokens_align+tokens_sft；direct用相同總量或時間。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
budget = 10000
plans = {
    "two_stage": {"alignment_tokens": 4000, "sft_tokens": 6000},
    "direct": {"alignment_tokens": 0, "sft_tokens": 10000},
}
print(plans)
assert all(sum(p.values()) == budget for p in plans.values())

**如何讀結果：**這是一張可執行配方的預算表，尚無實測收斂曲線；使用相同基模、seed與task測試。

**只改一件事：**只把對齊比例從40%改爲20%，保持總量。
